# Agent 2 - Multiple Statistical Tools

Goal: give the agent several tools and teach it to choose based on scenario.

```mermaid
flowchart TD
    Scenario[Scenario] --> Paired{Paired data?}
    Paired -- Yes --> PairTools[Paired t-test or Wilcoxon]
    Paired -- No --> Groups{How many groups?}
    Groups -- Two --> TwoTools[Welch, Student, or Mann-Whitney]
    Groups -- ThreePlus --> MultiTools[ANOVA or Kruskal-Wallis]
```

## Coding Agent Prompt

```text
Given a statistical scenario, route to one of this project's tests. Explain the routing rule, assumptions, and what additional information the agent should ask for before calling a tool.
```

In [ ]:
from typing import Annotated
import pandas as pd
from scipy import stats

def format_p_value(p: float) -> str:
    return f"{p:.4g}"

In [ ]:
def student_t_test(a, b):
    return stats.ttest_ind(a, b, equal_var=True)._asdict()

def mann_whitney(a, b):
    result = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"statistic": result.statistic, "p_value": result.pvalue}

def anova(*groups):
    result = stats.f_oneway(*groups)
    return {"statistic": result.statistic, "p_value": result.pvalue}

def choose_test(scenario: str) -> str:
    scenario = scenario.lower()
    if "three" in scenario or "multiple groups" in scenario or "intensity" in scenario:
        return "ANOVA or Kruskal-Wallis"
    if "non-normal" in scenario or "skew" in scenario:
        return "Mann-Whitney U"
    if "unequal variance" in scenario:
        return "Welch t-test"
    return "Welch t-test as a robust default for two independent groups"

choose_test("Compare event and non-event weeks with unequal variance")

## Agent Design Note

Tool selection should be grounded in assumptions:

- two independent groups: Welch or Student t-test
- two skewed groups: Mann-Whitney U
- three or more groups: ANOVA or Kruskal-Wallis
- paired before/after data: paired t-test or Wilcoxon signed-rank